# Cell 1 — Install Dependencies

In [ ]:
# ── Cell 1: Install all required libraries ──────────────────────────────────
!pip install -q supar openai sentencepiece wandb nltk spacy \
    huggingface_hub rouge-score bert-score numpy \
    language_tool_python==2.9.2 scikit-learn scipy tqdm \
    bitsandbytes transformers accelerate \
    Pillow ftfy regex torch==2.5.1 torchvision==0.20.1 \
    torchaudio==2.5.1 --extra-index-url https://download.pytorch.org/whl/cu124
print('All packages installed.')

# Cell 2 — Imports & Global Config

In [ ]:
# ── Cell 2: Imports & Global Config ─────────────────────────────────────────
import os, re, gc, time, json, csv, random, string, logging, warnings
import numpy as np
import torch
import nltk
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
from scipy.stats import entropy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import language_tool_python
import spacy

warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

# ── Download NLTK data ───────────────────────────────────────────────────────
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Paths — UPDATE THESE TO YOUR KAGGLE DATASET MOUNT PATHS ─────────────────
FULLTEXT_DIR  = '/kaggle/input/datasets/anshumanmoharana07/biolaysumm/Data/Short/fulltext'
SUMMARIES_DIR = '/kaggle/input/datasets/anshumanmoharana07/biolaysumm/Data/Short/summaries'
META_DIR      = '/kaggle/working/logs'
os.makedirs(META_DIR, exist_ok=True)

# ── Experiment hyper-params ──────────────────────────────────────────────────
POPULATION_SIZE   = 10      # Keep small for 2×T4 memory
NUM_ITER          = 5
NUM_CANDIDATES    = 5
NUM_OFFSPRING     = 2
MUTATION_PROB     = 0.5
NUM_COMPOSE       = 1
NUM_TOURNAMENTS   = 2
PATIENCE          = 5
NUM_TRAIN         = 50      # training subset size (keep low for speed)
NUM_TEST          = 50      # test subset size
BATCH_SIZE        = 1
NUM_SHOTS         = 0       # zero-shot (no examples in prompt)
SEED              = 3
TRAIN_SEED        = 42
BUDGET            = 4000    # max model calls before stopping
EDIT_OPERATIONS   = ['del', 'swap', 'sub', 'add']
META_NAME         = 'ga_results.txt'
WEIGHT_SUMM       = 0.6
WEIGHT_GRAM       = 0.4
MODEL_NAME        = 'Qwen/Qwen3-8B'

# Clear score log files
for fname in ['rouge_scores.txt', 'bert_scores.txt', 'bleu_scores.txt']:
    open(os.path.join(META_DIR, fname), 'w').close()

print('Config ready. META_DIR:', META_DIR)

# Cell 3 — CSV Logging Utilities


In [ ]:
# ── Cell 3: CSV Logging Utilities (NEW) ──────────────────────────────────────
#
# Three helpers used throughout the rest of the notebook:
#   - get_candidate_id(text)   -> stable short ID (C0000, C0001, ...) per
#                                  candidate string, so the same candidate
#                                  gets the same ID in every CSV file it shows
#                                  up in.
#   - log_predictions_csv(...) -> appends one row PER GENERATED SUMMARY (the
#                                  exact predictions ROUGE-L / BERTScore /
#                                  BLEU are computed from) to
#                                  iteration_<N>_summaries.csv
#   - log_event_csv(...)       -> appends one row per mutation / crossover
#                                  attempt to iteration_<N>_events.csv
#
# Iteration numbering convention (kept consistent with the score-curve plots
# later in the notebook, which already label the x-axis "0 = initial"):
#   iteration 0  -> the initial candidate (Cell 13)
#   iteration N  -> the N-th pass of the main GA loop (Cell 15), i.e. iter_idx + 1

CSV_LOG_DIR = os.path.join(META_DIR, "csv_logs")
os.makedirs(CSV_LOG_DIR, exist_ok=True)
print(f"CSV logs will be written to: {CSV_LOG_DIR}")

CURRENT_ITERATION = 0  # Cell 15 updates this at the start of every GA generation

_candidate_id_registry = {}
_candidate_id_counter = 0

def get_candidate_id(candidate_text):
    """Returns a stable short ID for a candidate string, reusing the same ID
    every time the same exact candidate text is seen again."""
    global _candidate_id_counter
    if candidate_text not in _candidate_id_registry:
        _candidate_id_registry[candidate_text] = f"C{_candidate_id_counter:04d}"
        _candidate_id_counter += 1
    return _candidate_id_registry[candidate_text]


def log_predictions_csv(iteration, candidate_text, ids, references,
                         predictions, rouge_scores, bert_scores, bleu_scores):
    """One row per generated summary used to compute ROUGE/BERT/BLEU for this
    candidate, written to csv_logs/iteration_<N>_summaries.csv."""
    candidate_id = get_candidate_id(candidate_text)
    csv_path = os.path.join(CSV_LOG_DIR, f"iteration_{iteration}_summaries.csv")
    file_exists = os.path.isfile(csv_path)
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["iteration", "candidate_id", "candidate_text", "doc_id",
                              "reference_summary", "generated_summary",
                              "rouge_l", "bert_score", "bleu"])
        for doc_id, ref, pred, r, b, bl in zip(ids, references, predictions,
                                                rouge_scores, bert_scores, bleu_scores):
            writer.writerow([iteration, candidate_id, candidate_text, doc_id, ref, pred, r, b, bl])
    return csv_path


def log_event_csv(iteration, event_type, edit_type="", parent_1="", parent_2="",
                   result_candidate="", accepted="", grammar_score="", notes=""):
    """One row per mutation/crossover attempt, written to
    csv_logs/iteration_<N>_events.csv. event_type is 'mutation' or 'crossover'."""
    csv_path = os.path.join(CSV_LOG_DIR, f"iteration_{iteration}_events.csv")
    file_exists = os.path.isfile(csv_path)
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["iteration", "event_type", "edit_type", "parent_1", "parent_2",
                              "result_candidate", "accepted", "grammar_score", "notes"])
        writer.writerow([iteration, event_type, edit_type, parent_1, parent_2,
                          result_candidate, accepted, grammar_score, notes])
    return csv_path

print("CSV logging utilities defined.")


# Cell 4 — GPU Utilities


In [ ]:
# ── Cell 4: GPU Utilities ────────────────────────────────────────────────────
def gpu_stats():
    for i in range(torch.cuda.device_count()):
        alloc   = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i)  / 1024**3
        print(f'  GPU {i}: {alloc:.2f} GB allocated | {reserved:.2f} GB reserved')

def clear_gpu():
    """Aggressively free GPU memory — call after every generation."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print('GPU count:', torch.cuda.device_count())
gpu_stats()

# Cell 5 — Load Qwen3-8B in 4-bit Quantization


In [ ]:
# ── Cell 5: Load Qwen3-8B (4-bit NF4 quantized, 2×T4 aware) ─────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',           # NF4 is best for LLMs
    bnb_4bit_use_double_quant=True,       # extra memory saving
    bnb_4bit_compute_dtype=torch.float16  # compute in fp16 on T4
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading model in 4-bit...')
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',            # splits across both T4s automatically
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
except RuntimeError as e:
    print(f'Error loading model: {e}')
    clear_gpu()
    raise

model.eval()
print('Model loaded successfully.')
gpu_stats()

# Generation config — keep outputs short (summaries)
GENERATION_ARGS = {
    'max_new_tokens': 150,
    'temperature': 0.1,
    'do_sample': False,
}

# Cell 6 — Dataset Loader (BioLaySumm .txt files)


In [ ]:
# ── Cell 6: Dataset Loader — reads paired .txt files ────────────────────────
import re as _re

def extract_index(filename):
    """
    Extract the numeric index from filenames like:
      multiclinsum_gs_en_42.txt     -> 42
      multiclinsum_gs_en_42_sum.txt -> 42
    """
    # Match the last number before optional '_sum' and '.txt'
    m = _re.search(r'_(\d+)(?:_sum)?\.txt$', filename)
    return int(m.group(1)) if m else None

def load_biolaysumm_pairs(fulltext_dir, summaries_dir):
    """
    Scan both directories, pair by index number.
    Returns a dict: {index: {'input': fulltext, 'output': summary}}
    """
    ft_files  = {extract_index(f): f for f in os.listdir(fulltext_dir)  if f.endswith('.txt') and '_sum' not in f}
    sum_files = {extract_index(f): f for f in os.listdir(summaries_dir) if f.endswith('_sum.txt')}

    common_ids = sorted(set(ft_files.keys()) & set(sum_files.keys()))
    print(f'Found {len(ft_files)} fulltexts, {len(sum_files)} summaries, {len(common_ids)} valid pairs.')

    pairs = {}
    for idx in common_ids:
        ft_path  = os.path.join(fulltext_dir,  ft_files[idx])
        sum_path = os.path.join(summaries_dir, sum_files[idx])
        with open(ft_path,  'r', encoding='utf-8', errors='ignore') as f:
            fulltext = f.read().strip()
        with open(sum_path, 'r', encoding='utf-8', errors='ignore') as f:
            summary  = f.read().strip()
        if fulltext and summary:
            pairs[idx] = {'input': fulltext, 'output': summary}
    print(f'Loaded {len(pairs)} non-empty pairs.')
    return pairs

# Load all pairs
ALL_PAIRS = load_biolaysumm_pairs(FULLTEXT_DIR, SUMMARIES_DIR)
ALL_IDS   = sorted(ALL_PAIRS.keys())

# Reproducible split
random.seed(TRAIN_SEED)
shuffled = ALL_IDS.copy()
random.shuffle(shuffled)
TEST_IDS  = shuffled[:NUM_TEST]
TRAIN_IDS = shuffled[NUM_TEST: NUM_TEST + NUM_TRAIN]
print(f'Train size: {len(TRAIN_IDS)} | Test size: {len(TEST_IDS)}')

# Cell 7 — Prompt Encoding & Inference


In [ ]:
# ── Cell 7: Prompt Encoding & Inference ─────────────────────────────────────

# Global evaluation cache to avoid re-running the same prompt
evaluation_cache = {}

# API call counter (tracks budget)
_api_call_count = 0

def build_messages(instruction, fulltext):
    """Build the chat-style message list for Qwen3."""
    user_content = f"{instruction}\n\nText:\n{fulltext}\n\nSummary:"
    return [
        {"role": "system", "content": "You are an expert biomedical summarizer."},
        {"role": "user",   "content": user_content}
    ]

@torch.inference_mode()
def run_model(messages, max_new_tokens=None):
    """Run a single inference call on the quantized Qwen3-8B."""
    global _api_call_count
    _api_call_count += 1

    tokens_to_gen = max_new_tokens or GENERATION_ARGS['max_new_tokens']

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False          # disable chain-of-thought for speed
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)

    # Truncate very long fulltexts to avoid OOM (T4 has 16 GB each)
    MAX_INPUT_TOKENS = 1024
    if inputs['input_ids'].shape[1] > MAX_INPUT_TOKENS:
        inputs['input_ids']      = inputs['input_ids'][:, :MAX_INPUT_TOKENS]
        inputs['attention_mask'] = inputs['attention_mask'][:, :MAX_INPUT_TOKENS]

    out_ids = model.generate(
        **inputs,
        max_new_tokens=tokens_to_gen,
        temperature=GENERATION_ARGS['temperature'],
        do_sample=GENERATION_ARGS['do_sample'],
        pad_token_id=tokenizer.eos_token_id
    )
    # Decode only the newly generated tokens
    new_tokens = out_ids[0][inputs['input_ids'].shape[1]:].tolist()
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Free activations immediately
    del inputs, out_ids
    torch.cuda.empty_cache()
    return result

def run_on_split(instruction, ids):
    """
    Run inference on a list of document IDs.
    Returns predictions, references, avg ROUGE-L, avg BERT, avg BLEU, and the
    per-instance ROUGE-L/BERT/BLEU lists (used for CSV logging by callers).
    """
    predictions, references = [], []
    for doc_id in tqdm(ids, desc='Inference', leave=False):
        pair = ALL_PAIRS[doc_id]
        msgs = build_messages(instruction, pair['input'])
        pred = run_model(msgs)
        predictions.append(pred)
        references.append(pair['output'])

    # ── ROUGE-L ──
    scorer_obj  = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer_obj.score(ref, pred)['rougeL'].fmeasure
                    for ref, pred in zip(references, predictions)]
    # ── BERTScore ──
    _, _, F1 = bert_score(predictions, references, lang='en', verbose=False,
                          device='cuda' if torch.cuda.is_available() else 'cpu',
                          batch_size=4)
    bert_scores = F1.tolist()
    # ── BLEU ──
    smoothie = SmoothingFunction().method4
    bleu_scores = [
        sentence_bleu([word_tokenize(ref.lower())], word_tokenize(pred.lower()),
                      smoothing_function=smoothie)
        for pred, ref in zip(predictions, references)
    ]

    avg_rouge = float(np.mean(rouge_scores)) * 100
    avg_bert  = float(np.mean(bert_scores))  * 100
    avg_bleu  = float(np.mean(bleu_scores))  * 100
    return predictions, references, avg_rouge, avg_bert, avg_bleu, rouge_scores, bert_scores, bleu_scores

print('Inference helpers defined.')

# Cell 8 — Objective Evaluation & Scoring


In [ ]:
# ── Cell 8: Objective Evaluation & Scoring ──────────────────────────────────
tool = language_tool_python.LanguageTool('en-US')

def language_tool_fallback(text):
    matches = tool.check(text)
    score = 100
    for m in matches:
        if m.ruleId.startswith('MORFOLOGIK_') or m.ruleId == 'UPPERCASE_SENTENCE_START':
            score -= 5
        elif 'AGREEMENT' in m.ruleId:
            score -= 15
        elif 'GRAMMAR' in m.ruleId or 'SENTENCE' in m.ruleId:
            score -= 20
        else:
            score -= 10
    return max(0, score)

def get_grammar_score(text):
    """Use the LLM to score grammar (0-100), fallback to LanguageTool."""
    sys_p = ("You are a strict grammar evaluator. "
             "Score grammar of the given text from 0 to 100. "
             "100 = perfect. 0 = very poor. Return ONLY a number.")
    usr_p = f'Score the grammar of this text:\n"""{text}"""'
    msgs  = [{"role": "system", "content": sys_p},
             {"role": "user",   "content": usr_p}]
    try:
        raw = run_model(msgs, max_new_tokens=5)
        nums = re.findall(r'\d+', raw)
        if nums:
            s = int(nums[0])
            if 0 <= s <= 100:
                return s
        return language_tool_fallback(text)
    except RuntimeError as e:
        if 'CUDA out of memory' in str(e):
            clear_gpu()
        return language_tool_fallback(text)

def evaluate_objectives(candidate, split='train', meta_file=None):
    """Full evaluation: summarization + grammar scores with caching."""
    if candidate in evaluation_cache:
        return evaluation_cache[candidate]

    ids = TRAIN_IDS if split == 'train' else TEST_IDS
    predictions, references, avg_rouge, avg_bert, avg_bleu, rouge_scores, bert_scores, bleu_scores = run_on_split(candidate, ids)

    summarization_score = WEIGHT_SUMM * avg_rouge + WEIGHT_GRAM * avg_bert
    grammar_score       = get_grammar_score(candidate)

    # --- CSV logging (NEW) --------------------------------------------------
    # Writes every generated summary used for the averages above to
    # csv_logs/iteration_<N>_summaries.csv, tagged with the candidate
    # instruction it came from.
    log_predictions_csv(
        CURRENT_ITERATION, candidate, ids, references,
        predictions, rouge_scores, bert_scores, bleu_scores
    )
    # -------------------------------------------------------------------------

    # Persist scores
    for fname, val, label in [
        ('rouge_scores.txt', avg_rouge, 'ROUGE'),
        ('bert_scores.txt',  avg_bert,  'BERT'),
        ('bleu_scores.txt',  avg_bleu,  'BLEU')
    ]:
        with open(os.path.join(META_DIR, fname), 'a') as f:
            f.write(f'Candidate: {candidate}\nAverage {label}: {val:.4f}\n\n')

    evaluation_cache[candidate] = (summarization_score, grammar_score,
                                   avg_rouge, avg_bert, avg_bleu)
    clear_gpu()
    return evaluation_cache[candidate]

print('Evaluation helpers defined.')

# Cell 9 — Grammar-Aware Phrase Mutation Helpers


In [ ]:
# ── Cell 9: Grammar-Aware Phrase Mutation Helpers ───────────────────────────
from supar import Parser as SuparParser

print('Loading SuPar CRF constituency parser...')
con_parser = SuparParser.load('crf-con-en')
print('Parser loaded.')

detokenizer = TreebankWordDetokenizer()

def detokenize(tokens):
    return detokenizer.detokenize(tokens)

def containsenglish(text):
    return bool(re.search('[a-zA-Z]', text))

def collect_leaves(parsed_tree):
    leaves = []
    for tree in parsed_tree:
        if not isinstance(tree, nltk.tree.Tree):
            continue
        if tree.label() == '_':
            leaves.append(detokenize(tree.leaves()))
            continue
        if check_child(tree):
            leaves.append(detokenize(tree.leaves()))
        else:
            leaves.extend(collect_leaves(tree))
    return leaves

def check_child(tree):
    count = sum(1 for s in tree if isinstance(s, nltk.tree.Tree) and s.label() == '_')
    return count >= len(tree) - count

def get_phrases(instruction):
    phrases = []
    exclude = {'he','she','it','they','i','we','me','him','her','them','us'}
    for sentence in sent_tokenize(instruction):
        parsed = con_parser.predict(word_tokenize(sentence), verbose=False).sentences[0].trees[0]
        leaves = collect_leaves(parsed)
        phrases.extend(leaves)
    normalized = []
    for p in phrases:
        if p not in string.punctuation and p.strip():
            norm = re.sub(r'\s+', ' ', p.strip())
            toks = word_tokenize(norm.lower())
            if len(toks) == 1 and toks[0] in exclude:
                continue
            normalized.append(norm)
    return normalized

def get_phrase_lookup(candidate):
    phrases = []
    for sentence in sent_tokenize(candidate):
        parsed = con_parser.predict(word_tokenize(sentence), verbose=False).sentences[0].trees[0]
        leaves = collect_leaves(parsed)
        phrases.extend(leaves)
    phrases = [detokenize(word_tokenize(p)) for p in phrases
               if p.strip() and not all(c in string.punctuation for c in p.strip())]
    return {i: p for i, p in enumerate(phrases)}

# ── Low-level edit operations ────────────────────────────────────────────────
def delete_phrase(candidate, phrase):
    if ' ' + phrase in candidate:
        return candidate.replace(' ' + phrase, ' ')
    if phrase + ' ' in candidate:
        return candidate.replace(phrase + ' ', ' ')
    return candidate.replace(phrase, '')

def add_phrase(candidate, phrase, after):
    if after == '':
        return phrase + ' ' + candidate
    if ' ' + after in candidate:
        return candidate.replace(' ' + after, ' ' + after + ' ' + phrase)
    if after + ' ' in candidate:
        return candidate.replace(after + ' ', after + ' ' + phrase + ' ')
    return candidate.replace(after, after + phrase)

def swap_phrases(candidate, p1, p2):
    # Use placeholder tokens to avoid double-replacement
    tmp = candidate
    tmp = tmp.replace(p1, '__P1__')
    tmp = tmp.replace(p2, '__P2__')
    tmp = tmp.replace('__P1__', p2)
    tmp = tmp.replace('__P2__', p1)
    return tmp

def substitute_phrase(candidate, phrase):
    """Use LLM to paraphrase a phrase and substitute it in the candidate."""
    sys_p = ('You are a paraphrasing expert. '
             'Paraphrase the given phrase so it fits in the instruction text. '
             'Return ONLY the paraphrased phrase, no extra text.')
    usr_p = (f'Instruction: {candidate}\n'
             f'Phrase to paraphrase: {phrase}\n'
             f'Paraphrased phrase:')
    msgs = [{"role": "system", "content": sys_p},
            {"role": "user",   "content": usr_p}]
    try:
        para = run_model(msgs, max_new_tokens=30).strip().strip('\'".')
        para = re.sub(r'^(Paraphrased phrase:)\s*', '', para).strip()
        if not para:
            return candidate
        new_cand = candidate.replace(phrase, para)
        if get_grammar_score(new_cand) < 10:
            return candidate
        return new_cand
    except Exception:
        return candidate

def perform_edit(edit, base_text, phrase_list, deleted_history):
    pl = phrase_list.copy()
    if edit == 'del':
        if not pl: return base_text, deleted_history
        chosen = random.choice(pl)
        deleted_history.append(chosen)
        return delete_phrase(base_text, chosen), deleted_history
    elif edit == 'swap':
        if len(pl) < 2: return base_text, deleted_history
        p1, p2 = random.sample(pl, 2)
        return swap_phrases(base_text, p1, p2), deleted_history
    elif edit == 'sub':
        if not pl: return base_text, deleted_history
        chosen = random.choice(pl)
        return substitute_phrase(base_text, chosen), deleted_history
    elif edit == 'add':
        if not deleted_history: return base_text, deleted_history
        phrase_to_add = random.choice(deleted_history)
        after = random.choice(pl) if pl else ''
        return add_phrase(base_text, phrase_to_add, after), deleted_history
    return base_text, deleted_history

def safe_mutation(edit, base_text, phrase_list, deleted_history, grammar_threshold=40):
    mutated, new_del = perform_edit(edit, base_text, phrase_list, deleted_history)
    if mutated == base_text:
        # perform_edit couldn't apply this edit (e.g. no phrases available) —
        # nothing actually happened, so there's no mutation event to log.
        return base_text, deleted_history
    gscore   = get_grammar_score(mutated)
    accepted = gscore >= grammar_threshold
    # --- CSV logging (NEW): record this mutation attempt -------------------
    log_event_csv(
        CURRENT_ITERATION, event_type="mutation", edit_type=edit,
        parent_1=base_text, result_candidate=mutated,
        accepted=accepted, grammar_score=gscore,
        notes="Mutation accepted" if accepted else "Mutation rejected (grammar below threshold)"
    )
    # -------------------------------------------------------------------------
    if not accepted:
        tqdm.write(f'  Mutation rejected (grammar={gscore})')
        return base_text, deleted_history
    return mutated, new_del

print('Mutation helpers defined.')

# Cell 10 — Crossover & Tournament Selection


In [ ]:
# ── Cell 10: Crossover & Tournament Selection ─────────────────────────────────

def tournament_selection(population, scores, k):
    """Pick k random candidates, return the best one."""
    idxs    = random.sample(range(len(population)), min(k, len(population)))
    best    = max(idxs, key=lambda i: scores[i])
    return population[best], scores[best]

def crossover(parent_1, parent_2, meta_file=None):
    """Phrase-level crossover between two instruction strings."""
    for attempt in range(3):
        try:
            lookup1 = get_phrase_lookup(parent_1)
            lookup2 = get_phrase_lookup(parent_2)
        except AttributeError:
            return parent_1, True

        p1_phrases = list(lookup1.values())
        p2_phrases = list(lookup2.values())
        if not p1_phrases or not p2_phrases:
            return parent_1, True

        total = min(len(p1_phrases), len(p2_phrases))
        n1    = random.randint(1, max(1, total - 1))
        n2    = total - n1

        random.shuffle(p1_phrases)
        random.shuffle(p2_phrases)
        offspring_phrases = p1_phrases[:n1] + p2_phrases[:n2]
        offspring_words   = []
        for ph in offspring_phrases:
            offspring_words.extend(word_tokenize(ph))
        offspring = detokenize(offspring_words)

        g        = get_grammar_score(offspring)
        accepted = containsenglish(offspring) and g >= 50
        # --- CSV logging (NEW): record this crossover attempt --------------
        log_event_csv(
            CURRENT_ITERATION, event_type="crossover", edit_type=f"attempt_{attempt+1}",
            parent_1=parent_1, parent_2=parent_2, result_candidate=offspring,
            accepted=accepted, grammar_score=g,
            notes="Crossover accepted" if accepted else "Crossover attempt rejected (grammar/non-English)"
        )
        # ---------------------------------------------------------------------
        if accepted:
            tqdm.write(f'  Crossover success (attempt {attempt+1}, grammar={g})')
            return offspring, False
        tqdm.write(f'  Crossover attempt {attempt+1} rejected (grammar={g})')

    # --- CSV logging (NEW): all 3 attempts failed, parent_1 is kept --------
    log_event_csv(
        CURRENT_ITERATION, event_type="crossover", edit_type="all_attempts_failed",
        parent_1=parent_1, parent_2=parent_2, result_candidate=parent_1,
        accepted=False, notes="All 3 crossover attempts failed — returning parent_1 unchanged"
    )
    # -------------------------------------------------------------------------
    return parent_1, True

print('Crossover and selection defined.')

# Cell 11 — NSGA-II: Non-Dominated Sorting & Crowding Distance


In [ ]:
# ── Cell 11: NSGA-II Sorting ─────────────────────────────────────────────────

def non_dominated_sorting(population_data):
    """
    population_data: list of (candidate, summ_score, gram_score, deleted_set)
    Returns list of fronts (each front = list of indices).
    """
    n = len(population_data)
    dom_count = [0] * n
    dom_set   = [[] for _ in range(n)]
    fronts    = []

    for i in range(n):
        for j in range(n):
            if i == j: continue
            pi_s, pi_g = population_data[i][1], population_data[i][2]
            pj_s, pj_g = population_data[j][1], population_data[j][2]
            if (pi_s >= pj_s and pi_g >= pj_g) and (pi_s > pj_s or pi_g > pj_g):
                dom_set[i].append(j)
            elif (pj_s >= pi_s and pj_g >= pi_g) and (pj_s > pi_s or pj_g > pi_g):
                dom_count[i] += 1

    front0 = [i for i in range(n) if dom_count[i] == 0]
    fronts.append(front0)
    current = front0
    while current:
        nxt = []
        for i in current:
            for j in dom_set[i]:
                dom_count[j] -= 1
                if dom_count[j] == 0:
                    nxt.append(j)
        if nxt:
            fronts.append(nxt)
        current = nxt
    return fronts

def crowding_distance(population_data, front):
    dist = {i: 0.0 for i in front}
    for obj_idx in [1, 2]:  # summarization=1, grammar=2
        sorted_f = sorted(front, key=lambda i: population_data[i][obj_idx])
        mn = population_data[sorted_f[0]][obj_idx]
        mx = population_data[sorted_f[-1]][obj_idx]
        dist[sorted_f[0]]  = float('inf')
        dist[sorted_f[-1]] = float('inf')
        for k in range(1, len(sorted_f) - 1):
            rng = mx - mn if mx != mn else 1e-9
            dist[sorted_f[k]] += (
                population_data[sorted_f[k+1]][obj_idx] -
                population_data[sorted_f[k-1]][obj_idx]
            ) / rng
    return dist

print('NSGA-II helpers defined.')

# Cell 12 — Plotting Utilities


In [ ]:
# ── Cell 12: Plotting Utilities ──────────────────────────────────────────────

def plot_pareto_front(population_data, fronts, iteration, meta_dir, final=False):
    plt.figure(figsize=(8, 6))
    s_all = [d[1] for d in population_data]
    g_all = [d[2] for d in population_data]
    plt.scatter(s_all, g_all, c='gray', alpha=0.4, label='All')
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for fi, front in enumerate(fronts[:len(colors)]):
        xs = [population_data[i][1] for i in front]
        ys = [population_data[i][2] for i in front]
        lbl = 'Pareto Front' if fi == 0 else f'Front {fi+1}'
        plt.scatter(xs, ys, c=colors[fi], label=lbl)
        sx = sorted(range(len(xs)), key=lambda k: xs[k])
        plt.plot([xs[k] for k in sx], [ys[k] for k in sx],
                 c=colors[fi], linestyle='--')
    plt.xlabel('Summarization Score')
    plt.ylabel('Grammar Score')
    title = 'Pareto Front (Final)' if final else f'Pareto Front — Iter {iteration}'
    plt.title(title)
    plt.legend(); plt.grid(True)
    fname = 'pareto_final.png' if final else f'pareto_iter_{iteration}.png'
    path  = os.path.join(meta_dir, fname)
    plt.savefig(path, dpi=80, bbox_inches='tight')
    plt.close()
    return path

def plot_score_curves(meta_dir, rouge, bert, bleu, summ, gram):
    iters = list(range(len(rouge)))
    metrics = [
        (rouge, 'ROUGE-L',        'best_rouge.png'),
        (bert,  'BERT Score',     'best_bert.png'),
        (bleu,  'BLEU',           'best_bleu.png'),
        (summ,  'Summarization',  'best_summ.png'),
        (gram,  'Grammar',        'best_grammar.png'),
    ]
    for values, ylabel, fname in metrics:
        plt.figure(figsize=(8, 5))
        plt.plot(iters, values, marker='o')
        plt.xlabel('Iteration (0 = initial)')
        plt.ylabel(ylabel)
        plt.title(f'Best Candidate {ylabel} vs Iteration')
        plt.grid(True); plt.xticks(iters)
        plt.savefig(os.path.join(meta_dir, fname), dpi=80, bbox_inches='tight')
        plt.close()
    print(f'Score curve plots saved to {meta_dir}')

print('Plotting utilities defined.')

# Cell 13 — WandB Setup (Optional)


In [ ]:
# ── Cell 13: WandB Setup (set key or leave blank to skip) ────────────────────
WANDB_KEY     = ''        # paste your key here, or leave blank
WANDB_PROJECT = 'BioLaySumm-GA-Qwen3-4bit'
USE_WANDB     = False

if WANDB_KEY:
    try:
        import wandb
        wandb.login(key=WANDB_KEY)
        wandb.init(project=WANDB_PROJECT)
        wandb.config.update({
            'population_size': POPULATION_SIZE,
            'num_iter': NUM_ITER,
            'mutation_prob': MUTATION_PROB,
            'num_train': NUM_TRAIN,
            'num_test': NUM_TEST,
            'model': MODEL_NAME,
            'quantization': '4bit-NF4'
        })
        USE_WANDB = True
        print('WandB ready.')
    except Exception as e:
        print(f'WandB skipped: {e}')
else:
    print('WandB skipped (no key). Logs saved locally to', META_DIR)

def wlog(d):
    if USE_WANDB:
        wandb.log(d)

# Cell 14 — Initialize Population & Evaluate Initial Candidate


In [ ]:
# ── Cell 14: Initialize Population ──────────────────────────────────────────
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)
random.seed(TRAIN_SEED)

meta_path = os.path.join(META_DIR, META_NAME)
meta_file = open(meta_path, 'w+')

# Initial instruction (task-agnostic)
INITIAL_INSTRUCTION = (
    "Summarize the given document in clear and grammatically correct English while preserving the key information."
)

CURRENT_ITERATION = 0  # this evaluation belongs to iteration 0 (the baseline)

print('Evaluating initial candidate...')
init_summ, init_gram, init_rouge, init_bert, init_bleu = evaluate_objectives(
    INITIAL_INSTRUCTION, split='train', meta_file=meta_file
)

print(f'Initial Instruction: {INITIAL_INSTRUCTION}')
print(f'Summarization={init_summ:.2f} | Grammar={init_gram} | '
      f'ROUGE={init_rouge:.2f} | BERT={init_bert:.2f} | BLEU={init_bleu:.2f}')

meta_file.write(f'Initial Instruction: {INITIAL_INSTRUCTION}\n')
meta_file.write(f'Objectives: summ={init_summ:.4f} gram={init_gram} '
                f'rouge={init_rouge:.4f} bert={init_bert:.4f} bleu={init_bleu:.4f}\n\n')

wlog({
    'Initial/Summarization': init_summ, 'Initial/Grammar': init_gram,
    'Initial/ROUGE': init_rouge, 'Initial/BERT': init_bert, 'Initial/BLEU': init_bleu
})

# Initialise population — all start from the same instruction
W_candidates  = [INITIAL_INSTRUCTION] * POPULATION_SIZE
W_objectives  = [(init_summ, init_gram)] * POPULATION_SIZE
W_deletesets  = [[] for _ in range(POPULATION_SIZE)]

# Score tracking across iterations
best_rouge_scores = [init_rouge]
best_bert_scores  = [init_bert]
best_bleu_scores  = [init_bleu]
best_summ_scores  = [init_summ]
best_gram_scores  = [init_gram]

best_candidate      = INITIAL_INSTRUCTION
best_patience_ctr   = 0
edit_ops            = EDIT_OPERATIONS.copy()

# --- Global-best tracker (NEW) -----------------------------------------------
# Tracks the single highest-weighted-score candidate seen across ALL
# iterations (not just the last one that ran). Purely a passive observer —
# it reads scores the algorithm already computed and never feeds back into
# selection, mutation, crossover, or the NSGA-II population at all.
global_best_candidate      = INITIAL_INSTRUCTION
global_best_weighted_score = WEIGHT_SUMM * init_summ + WEIGHT_GRAM * init_gram
global_best_objectives     = (init_summ, init_gram)
global_best_scores         = (init_rouge, init_bert, init_bleu, init_summ, init_gram)
global_best_iteration      = 0
# -------------------------------------------------------------------------------

clear_gpu()
gpu_stats()
print('Population initialised.')

# Cell 15 — Main GA Evolutionary Loop


In [ ]:
# ── Cell 15: Main GA Evolutionary Loop ──────────────────────────────────────
start_time = time.time()

for iter_idx in range(NUM_ITER):
    # CURRENT_ITERATION drives every CSV filename written this generation
    # (iteration_<N>_summaries.csv / iteration_<N>_events.csv). Convention:
    # iteration 0 = baseline (Cell 14), so the first GA pass here is iteration 1.
    CURRENT_ITERATION = iter_idx + 1

    tqdm.write(f'\n══════ Iteration {iter_idx} / {NUM_ITER-1} ══════')
    meta_file.write(f'\n=== Iteration {iter_idx} ===\n')

    # ── Step 1: Crossover ────────────────────────────────────────────────────
    new_candidates = W_candidates.copy()
    new_objectives = W_objectives.copy()
    new_deletesets = W_deletesets.copy()

    for j in range(NUM_OFFSPRING):
        p1, _ = tournament_selection(W_candidates,
                                     [o[0] for o in W_objectives], NUM_TOURNAMENTS)
        p2, _ = tournament_selection(W_candidates,
                                     [o[0] for o in W_objectives], NUM_TOURNAMENTS)
        tqdm.write(f'  Crossover {j}: parent1="{p1[:60]}..." parent2="{p2[:60]}..."')
        meta_file.write(f'Parent1: {p1}\nParent2: {p2}\n')

        offspring, err = crossover(p1, p2, meta_file)
        if err or not containsenglish(offspring):
            tqdm.write('  Crossover skipped.')
            continue
        tqdm.write(f'  Offspring: "{offspring[:80]}"')
        meta_file.write(f'Offspring: {offspring}\n')
        new_candidates.append(offspring)
        new_deletesets.append(W_deletesets[W_candidates.index(p1)])

    # ── Step 2: Mutation ─────────────────────────────────────────────────────
    for idx in range(min(POPULATION_SIZE, len(new_candidates))):
        if MUTATION_PROB <= np.random.random():
            continue
        base = new_candidates[idx]
        try:
            phrases = get_phrases(base)
        except AttributeError:
            tqdm.write(f'  Mutation skipped for candidate {idx} (parse error)')
            continue
        if not phrases:
            continue

        # Remove 'add' if there's nothing to add back
        local_ops = edit_ops.copy()
        if not new_deletesets[idx] and 'add' in local_ops:
            local_ops.remove('add')

        sampled_edits = np.random.choice(local_ops, NUM_CANDIDATES)
        tqdm.write(f'  Mutation edits for candidate {idx}: {sampled_edits}')
        for op in sampled_edits:
            mutated, new_deletesets[idx] = safe_mutation(
                op, base, phrases.copy(), new_deletesets[idx],
                grammar_threshold=40
            )
            if mutated != base:
                new_candidates[idx] = mutated
                tqdm.write(f'  Applied "{op}": "{mutated[:80]}"')
                meta_file.write(f'Mutation({op}): {mutated}\n')
                break

    # ── Step 3: Evaluate all candidates ─────────────────────────────────────
    tqdm.write('  Evaluating candidates...')
    new_objectives_full = []
    candidate_scores    = []
    for cand in tqdm(new_candidates, desc='Eval', leave=False):
        s, g, r, b, bl = evaluate_objectives(cand, split='train',
                                              meta_file=meta_file)
        new_objectives_full.append((s, g))
        candidate_scores.append((r, b, bl, s, g))

    # ── Step 4: NSGA-II selection ────────────────────────────────────────────
    combined = list(zip(new_candidates, new_objectives_full, new_deletesets))
    pop_data = [(c, o[0], o[1], d) for c, o, d in combined]

    fronts = non_dominated_sorting(pop_data)
    tqdm.write(f'  Fronts: {[len(f) for f in fronts]}')
    meta_file.write(f'Fronts sizes: {[len(f) for f in fronts]}\n')

    plot_pareto_front(pop_data, fronts, iter_idx, META_DIR)

    # Fill next generation
    selected_idx = []
    remaining    = POPULATION_SIZE
    for front in fronts:
        if len(front) <= remaining:
            selected_idx.extend(front)
            remaining -= len(front)
        else:
            dist = crowding_distance(pop_data, front)
            sf   = sorted(front, key=lambda i: dist[i], reverse=True)
            selected_idx.extend(sf[:remaining])
            remaining = 0
        if remaining == 0:
            break

    W_candidates = [pop_data[i][0] for i in selected_idx]
    W_objectives = [(pop_data[i][1], pop_data[i][2]) for i in selected_idx]
    W_deletesets = [pop_data[i][3] for i in selected_idx]

    # ── Step 5: Track best candidate ────────────────────────────────────────
    pareto_front = fronts[0]
    best_idx = max(
        pareto_front,
        key=lambda i: WEIGHT_SUMM * pop_data[i][1] + WEIGHT_GRAM * pop_data[i][2]
    )
    iter_best      = pop_data[best_idx][0]
    iter_best_objs = (pop_data[best_idx][1], pop_data[best_idx][2])
    iter_scores    = candidate_scores[best_idx]  # (rouge, bert, bleu, summ, gram)

    # --- Global-best tracker (NEW) ------------------------------------------
    # Compares this iteration's winner against the best ever seen so far.
    # Read-only with respect to the algorithm: doesn't touch W_candidates,
    # W_objectives, fronts, or anything NSGA-II uses for the next generation.
    iteration_weighted_score = WEIGHT_SUMM*iter_best_objs[0] + WEIGHT_GRAM*iter_best_objs[1]
    if iteration_weighted_score > global_best_weighted_score:
        global_best_candidate      = iter_best
        global_best_weighted_score = iteration_weighted_score
        global_best_objectives     = iter_best_objs
        global_best_scores         = iter_scores
        global_best_iteration      = iter_idx + 1
    # -----------------------------------------------------------------------------

    best_rouge_scores.append(iter_scores[0])
    best_bert_scores.append(iter_scores[1])
    best_bleu_scores.append(iter_scores[2])
    best_summ_scores.append(iter_scores[3])
    best_gram_scores.append(iter_scores[4])

    tqdm.write(f'  Best this iter: "{iter_best[:100]}"')
    tqdm.write(f'  Objectives: summ={iter_best_objs[0]:.2f} | gram={iter_best_objs[1]}')
    meta_file.write(f'Best: {iter_best}\nObjectives: {iter_best_objs}\n')

    wlog({
        f'iter/best_rouge': iter_scores[0],
        f'iter/best_bert':  iter_scores[1],
        f'iter/best_bleu':  iter_scores[2],
        f'iter/best_summ':  iter_scores[3],
        f'iter/best_gram':  iter_scores[4],
        'iteration': iter_idx,
        'global_best/weighted_score': global_best_weighted_score,
        'global_best/found_at_iteration': global_best_iteration
    })

    # ── Patience check ───────────────────────────────────────────────────────
    if iter_best == best_candidate:
        best_patience_ctr += 1
    else:
        best_candidate    = iter_best
        best_patience_ctr = 0

    if best_patience_ctr >= PATIENCE:
        tqdm.write('  Early stop: patience exhausted.')
        meta_file.write('Early stop: patience.\n')
        break
    if _api_call_count >= BUDGET:
        tqdm.write(f'  Early stop: budget ({BUDGET} calls) reached.')
        meta_file.write('Early stop: budget.\n')
        break

    # ── GPU housekeeping after every generation ──────────────────────────────
    clear_gpu()
    gpu_stats()

print(f'\nGA finished. Total model calls: {_api_call_count}')
print(f'Best instruction found (last iteration that ran):\n  {best_candidate}')
print(f'Global best instruction (best across ALL iterations, found at iteration {global_best_iteration}):\n  {global_best_candidate}')
meta_file.write(f'\nTotal API calls: {_api_call_count}\n')
meta_file.write(f'Final best instruction (last iteration): {best_candidate}\n')
meta_file.write(f'Global best instruction (iteration {global_best_iteration}): {global_best_candidate}\n')
meta_file.write(f'Global best weighted score: {global_best_weighted_score:.4f}\n')

# Cell 16 — Final Pareto Plot & Score Curves


In [ ]:
# ── Cell 16: Final Pareto Plot & Score Curves ────────────────────────────────
plot_pareto_front(pop_data, fronts, iter_idx, META_DIR, final=True)

plot_score_curves(
    META_DIR,
    best_rouge_scores,
    best_bert_scores,
    best_bleu_scores,
    best_summ_scores,
    best_gram_scores
)

print('All plots saved to', META_DIR)

# --- Global-best tracker (NEW): final report -------------------------------
print(f"\n========== GLOBAL BEST PROMPT (best across ALL iterations, found at iteration {global_best_iteration}) ==========")
print(global_best_candidate)
print("===================================================================================================")
print(f"ROUGE Score             : {global_best_scores[0]:.4f}")
print(f"BERT Score               : {global_best_scores[1]:.4f}")
print(f"BLEU Score                : {global_best_scores[2]:.4f}")
print(f"Summarization Score   : {global_best_objectives[0]:.4f}")
print(f"Grammar Score            : {global_best_objectives[1]:.4f}")
print(f"Weighted Score (0.6/0.4): {global_best_weighted_score:.4f}")
if global_best_candidate == best_candidate:
    print("(Same as the last-iteration result — the search was still improving right up to the end.)")
else:
    print(f"(Different from the last-iteration result — iteration {global_best_iteration} outperformed the final iteration.)")

meta_file.write(f"\nGlobal Best Candidate (found at iteration {global_best_iteration}):\t {global_best_candidate}\n")
meta_file.write(f"Global Best Objectives (Summarization, Grammar):\t {str(global_best_objectives)}\n")
meta_file.write(f"Global Best Weighted Score:\t {global_best_weighted_score:.4f}\n")

wlog({
    'global_best/candidate': global_best_candidate,
    'global_best/iteration': global_best_iteration,
    'global_best/summarization': global_best_objectives[0],
    'global_best/grammar': global_best_objectives[1],
    'global_best/weighted_score': global_best_weighted_score
})

# --- CSV logging (NEW): one-row audit record of the global best ------------
global_best_id = get_candidate_id(global_best_candidate)
global_best_csv_path = os.path.join(CSV_LOG_DIR, "global_best.csv")
with open(global_best_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["candidate_id", "candidate_text", "found_at_iteration",
                      "rouge_l", "bert_score", "bleu",
                      "summarization_score", "grammar_score", "weighted_score"])
    writer.writerow([global_best_id, global_best_candidate, global_best_iteration,
                      global_best_scores[0], global_best_scores[1], global_best_scores[2],
                      global_best_objectives[0], global_best_objectives[1], global_best_weighted_score])
print(f"\nGlobal best record written: {global_best_csv_path}")
print(f"(candidate_id {global_best_id} — look this up in iteration_{global_best_iteration}_summaries.csv "
      f"to see the exact generated summaries behind this score)")
# -----------------------------------------------------------------------------

# --- CSV logging (NEW): list per-iteration files + build combined files ----
# Per-iteration files already exist at this point:
#   csv_logs/iteration_0_summaries.csv, iteration_1_summaries.csv, ...
#   csv_logs/iteration_0_events.csv,    iteration_1_events.csv,    ...
# (only iterations where a mutation/crossover actually fired will have an
# events file — that's expected, not a bug)
print(f"\nPer-iteration CSV logs saved in: {CSV_LOG_DIR}")
for fname in sorted(os.listdir(CSV_LOG_DIR)):
    print("  -", fname)

import glob
def _combine_csvs(pattern, out_name):
    files = sorted(
        glob.glob(os.path.join(CSV_LOG_DIR, pattern)),
        key=lambda p: int(re.search(r'iteration_(\d+)_', p).group(1))
    )
    if not files:
        return
    out_path = os.path.join(CSV_LOG_DIR, out_name)
    with open(out_path, 'w', newline='', encoding='utf-8') as out_f:
        writer = None
        for fp in files:
            with open(fp, 'r', newline='', encoding='utf-8') as in_f:
                reader = csv.reader(in_f)
                header = next(reader)
                if writer is None:
                    writer = csv.writer(out_f)
                    writer.writerow(header)
                for row in reader:
                    writer.writerow(row)
    print(f"Combined file written: {out_path}")

_combine_csvs("iteration_*_summaries.csv", "all_iterations_summaries.csv")
_combine_csvs("iteration_*_events.csv",    "all_iterations_events.csv")
# -----------------------------------------------------------------------------

# Cell 17 — Test Set Evaluation of Best Prompt


In [ ]:
# ── Cell 17: Test Set Evaluation of Best Prompt ──────────────────────────────
# ── Cell 17: Test Set Evaluation of Best Prompt ──────────────────────────────
print('Running final evaluation on TEST set...')
clear_gpu()

def _evaluate_on_test(candidate_text, label):
    """Runs candidate_text on TEST_IDS and writes predictions to both JSON
    and CSV. label is a short tag ('last_iteration' or 'global_best') used
    in filenames so the two evaluations don't overwrite each other."""
    preds, refs, rouge_s, bert_s, bleu_s, rouge_list, bert_list, bleu_list = run_on_split(
        candidate_text, TEST_IDS
    )
    summ = WEIGHT_SUMM * rouge_s + WEIGHT_GRAM * bert_s

    json_path = os.path.join(META_DIR, f'test_predictions_{label}.json')
    with open(json_path, 'w') as pf:
        json.dump({
            'instruction':   candidate_text,
            'predictions':   preds,
            'references':    refs,
            'ids':           TEST_IDS,
            'rouge':         rouge_s,
            'bert':          bert_s,
            'bleu':          bleu_s,
            'summarization': summ
        }, pf, indent=2)

    # --- CSV logging (NEW): per-instance test-set predictions --------------
    csv_path = os.path.join(CSV_LOG_DIR, f'test_set_predictions_{label}.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["candidate_id", "candidate_text", "doc_id",
                          "reference_summary", "generated_summary",
                          "rouge_l", "bert_score", "bleu"])
        cand_id = get_candidate_id(candidate_text)
        for doc_id, ref, pred, r, b, bl in zip(TEST_IDS, refs, preds, rouge_list, bert_list, bleu_list):
            writer.writerow([cand_id, candidate_text, doc_id, ref, pred, r, b, bl])
    # -------------------------------------------------------------------------

    return preds, refs, rouge_s, bert_s, bleu_s, summ, json_path, csv_path


test_preds, test_refs, test_rouge, test_bert, test_bleu, test_summ, pred_path, test_csv_path = \
    _evaluate_on_test(best_candidate, 'last_iteration')

print(f'\n── Final Test Scores (last-iteration best) ──')
print(f'  Best Instruction : {best_candidate}')
print(f'  ROUGE-L          : {test_rouge:.4f}')
print(f'  BERT Score       : {test_bert:.4f}')
print(f'  BLEU             : {test_bleu:.4f}')
print(f'  Summarization    : {test_summ:.4f}')
print(f'Predictions saved to {pred_path}')
print(f'Predictions CSV saved to {test_csv_path}')

meta_file.write(f'\n=== Test Results (last-iteration best) ===\n')
meta_file.write(f'ROUGE={test_rouge:.4f} BERT={test_bert:.4f} BLEU={test_bleu:.4f}\n')
meta_file.write(f'Instruction: {best_candidate}\n')

wlog({'Test/ROUGE': test_rouge, 'Test/BERT': test_bert,
      'Test/BLEU': test_bleu, 'Test/Summarization': test_summ,
      'Final/Best_Instruction': best_candidate})

# --- Global-best tracker (NEW): also test the true global-best candidate ---
# (skipped if it's literally the same candidate as best_candidate, to avoid
# a redundant pass over the test set)
if global_best_candidate != best_candidate:
    gb_preds, gb_refs, gb_rouge, gb_bert, gb_bleu, gb_summ, gb_pred_path, gb_csv_path = \
        _evaluate_on_test(global_best_candidate, 'global_best')

    print(f'\n── Final Test Scores (global best, found at iteration {global_best_iteration}) ──')
    print(f'  Best Instruction : {global_best_candidate}')
    print(f'  ROUGE-L          : {gb_rouge:.4f}')
    print(f'  BERT Score       : {gb_bert:.4f}')
    print(f'  BLEU             : {gb_bleu:.4f}')
    print(f'  Summarization    : {gb_summ:.4f}')
    print(f'Predictions saved to {gb_pred_path}')
    print(f'Predictions CSV saved to {gb_csv_path}')

    meta_file.write(f'\n=== Test Results (global best, iteration {global_best_iteration}) ===\n')
    meta_file.write(f'ROUGE={gb_rouge:.4f} BERT={gb_bert:.4f} BLEU={gb_bleu:.4f}\n')
    meta_file.write(f'Instruction: {global_best_candidate}\n')

    wlog({'Test/GlobalBest_ROUGE': gb_rouge, 'Test/GlobalBest_BERT': gb_bert,
          'Test/GlobalBest_BLEU': gb_bleu, 'Test/GlobalBest_Summarization': gb_summ,
          'Final/GlobalBest_Instruction': global_best_candidate})
else:
    print('\n(Global best is the same candidate as the last-iteration best — no separate test run needed.)')
# -----------------------------------------------------------------------------

meta_file.close()

if USE_WANDB:
    wandb.save(meta_path)
    wandb.finish()

clear_gpu()
print('Done.')